# Pre-parsed value extraction

**Difficulty:** Beginner

A regex finds candidate values first; `Choice` only ever picks among the spans the regex already found. The value you get back is one of those spans, copied unchanged — it can't invent a value or transpose a digit, because it was never allowed to generate one.

Ask your agent if anything's unclear.

<details>
  <summary><strong>Primary source (trimmed):</strong> docs.typesafe.ai/cookbooks/pre_parsed_value_extraction_cookbook (click to expand)</summary>

# Pre-parsed value extraction

> Uses regexes to find candidate emails, phone numbers, and amounts, then has TypeSafe select the requested span so code can normalize a verbatim value.

`Choice` picks one of the options you hand it — so the candidates have to be found first. Three steps:

1. A regex finds the candidate values in the text. Tune it to over-find.
2. TypeSafe picks which candidate the question is asking for, and reads off any attribute code needs downstream (currency, country, credit vs. charge).
3. Code copies the picked value and normalizes it.

Because TypeSafe only ever chooses among the spans the regex found, the answer is always one of those spans, copied unchanged.

## Email: pick the right address by role

```
From: Dana Whit <dana.whit@acme-corp.com>
To: billing@acme-corp.com
Cc: orders@acme-corp.com
Reply-To: dana.personal@gmail.com

Hi team - please don't use the billing alias for this one. Send my receipt to my
personal address instead. Thanks, Dana.
```

```
candidates : ['dana.whit@acme-corp.com', 'billing@acme-corp.com', 'orders@acme-corp.com', 'dana.personal@gmail.com']
receipt -> : dana.personal@gmail.com      (conf 0.98)
sender  -> : dana.whit@acme-corp.com      (conf 1.00)
```

`receipt` is the personal Gmail on the `Reply-To:` line — what the message body actually asks for, not the `To:` billing alias. Both answers are verbatim copies of regex matches, only lowercased in code.

## Phone: pick the mobile, normalize to E.164

Three numbers, none carrying a country code (`(415) 555-0199`, `(415) 555-0142`, `(415) 555-0177`). TypeSafe picks which one is "the direct mobile / cell number" (from context: "call the cell if it's urgent") and separately reads which country the office is in from the surrounding text (`US`, at confidence 0.90). Neither answer comes from the digits themselves — a phone-number library then combines the picked number and the read country into E.164: `+14155550177`.

## Money: pick the amount, classify the currency, flag credit vs charge

```
Invoice INV-2087.
Subtotal: $1,200.00
Sales tax: $115.50
Total due: $1,315.50
A $50.00 courtesy credit from last month has already been applied.
```

```
total due : $1,315.50  -> 1315.50 USD (charge, P(credit)=0.01)
credit    : $50.00     -> 50.00 USD (credit, P(credit)=0.99)

candidates : ['$1,200.00', '$115.50', '$1,315.50', '$50.00']
```

Three separate judgments over the same four candidates: `Choice` picks the total, `Choice` picks the credit, a `Choice` classifies the currency, and a `Noul` per picked amount asks whether it's a credit rather than a charge (0.01 for the total, 0.99 for the credit) — so code knows the sign of each parsed `Decimal`.

> `to_decimal` assumes the comma groups thousands and the dot is the decimal point — true for `$1,315.50`, backwards for `€1.315,50`. Ask a `Noul` which convention the document uses, and branch on it, rather than assuming.

## Two limits

- A `Choice` question allows at most 255 options — with more candidates, narrow in two stages: pick the section first, then the span inside it.
- Finding candidates is the part that takes real work. Emails, phones, and amounts have regexes that cover them well; a *name* doesn't — its candidates have to come from a roster you already have, or a named-entity recognizer, or an LLM that proposes them. TypeSafe only ever picks among whatever candidates you hand it.

> The full cookbook also covers the phone/E.164 normalization end-to-end with the `phonenumbers` library — trimmed here to email and money, since a phone-number library is an extra dependency this repo doesn't need for the underlying point. Read the full cookbook at [docs.typesafe.ai/cookbooks/pre_parsed_value_extraction_cookbook](https://docs.typesafe.ai/cookbooks/pre_parsed_value_extraction_cookbook) any time.

</details>

## The core idea

This is `Choice` (lesson 2) with a specific discipline: the options aren't written by hand, they're *found* — by a regex, by a roster, by anything code can enumerate — and the model's job shrinks to picking among exactly those. That's a stronger guarantee than a normal `Choice` gives you: not just "one of a small labeled set," but "one of the *literal spans already present in the source text*," so there's no path to a fabricated or digit-transposed value. Finding good candidates is real work; picking among them is what TypeSafe is for.

In [ ]:
import { client } from "../../assets/typesafe-client.ts";
import { choice, noul } from "npm:@typesafe-ai/sdk";

const EMAIL_RE = /[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}/g;
const MONEY_RE = /[$€£¥]\s?\d[\d,]*(?:\.\d{2})?/g;

function find(pattern: RegExp, text: string): string[] {
  const seen = new Set<string>();
  const out: string[] = [];
  for (const match of text.matchAll(pattern)) {
    const span = match[0].trim();
    if (span && !seen.has(span)) {
      seen.add(span);
      out.push(span);
    }
  }
  return out;
}

const NONE = "none";

async function pick(document: string, candidates: string[], question: string) {
  const criteria: Record<string, string | null> = Object.fromEntries(
    candidates.map((c) => [c, null]),
  );
  criteria[NONE] = "None of these is the requested value.";
  const response = await client.systemOne({
    state: document,
    questions: { pick: choice(question, criteria) },
  });
  return { choice: response.answers.pick.choice, confidence: response.answers.pick.confidence };
}

async function classify(document: string, question: string, options: string[]) {
  const criteria = Object.fromEntries(options.map((o) => [o, null]));
  const response = await client.systemOne({
    state: document,
    questions: { q: choice(question, criteria) },
  });
  return { choice: response.answers.q.choice, confidence: response.answers.q.confidence };
}

async function isTrue(document: string, question: string): Promise<number> {
  const response = await client.systemOne({ state: document, questions: { q: noul(question) } });
  return response.answers.q.noul;
}

// --- Email: pick the right address by role ---
const EMAIL_DOC = `From: Dana Whit <dana.whit@acme-corp.com>
To: billing@acme-corp.com
Cc: orders@acme-corp.com
Reply-To: dana.personal@gmail.com

Hi team - please don't use the billing alias for this one. Send my receipt to my
personal address instead. Thanks, Dana.`;

const emails = find(EMAIL_RE, EMAIL_DOC);
const receipt = await pick(
  EMAIL_DOC,
  emails,
  "Which email address does the sender want their receipt sent to?",
);
const sender = await pick(
  EMAIL_DOC,
  emails,
  "Which email address did this message come from (the From line)?",
);

console.log("email candidates:", emails);
console.log(`receipt -> ${receipt.choice.toLowerCase()} (conf ${receipt.confidence.toFixed(2)})`);
console.log(`sender  -> ${sender.choice.toLowerCase()} (conf ${sender.confidence.toFixed(2)})`);

// --- Money: pick the amount, classify the currency, flag credit vs charge ---
const MONEY_DOC = `Invoice INV-2087.
Subtotal: $1,200.00
Sales tax: $115.50
Total due: $1,315.50
A $50.00 courtesy credit from last month has already been applied.`;

const amounts = find(MONEY_RE, MONEY_DOC);
const currency = await classify(MONEY_DOC, "What currency are these amounts in?", [
  "USD",
  "EUR",
  "GBP",
  "JPY",
  "CAD",
]);
const total = await pick(MONEY_DOC, amounts, "Which amount is the total the customer must pay?");
const credit = await pick(
  MONEY_DOC,
  amounts,
  "Which amount is the courtesy credit that was applied?",
);

function toDecimal(value: string): number {
  return Number(value.replace(/[^\d.]/g, ""));
}

const pairs: [string, { choice: string; confidence: number }][] = [
  ["total due", total],
  ["credit", credit],
];
for (const [label, chosen] of pairs) {
  const isCredit = await isTrue(
    MONEY_DOC,
    `Is the amount ${chosen.choice} a credit or refund to the customer, not a charge?`,
  );
  const kind = isCredit > 0.5 ? "credit" : "charge";
  console.log(
    `${label}: ${chosen.choice} -> ${toDecimal(chosen.choice)} ${currency.choice} (${kind}, P(credit)=${
      isCredit.toFixed(2)
    })`,
  );
}
console.log("money candidates:", amounts);

## What just happened

Two independent extractions, same three-step shape. For email: four addresses found, `receipt` should land on the personal Gmail (what the body actually asks for), `sender` on the `From:` address — neither answer can be anything the regex didn't already find. For money: four amounts found, `total` and `credit` should land on `$1,315.50` and `$50.00` respectively, with the follow-up `Noul` correctly flagging the credit as `P(credit)` near 1 and the total as near 0.

**Try it yourself:** add a fifth, decoy amount to `MONEY_DOC` that looks like a total but isn't (e.g. a "previous balance: $980.00" line), and predict whether `total`'s pick changes — the regex will find it as a candidate either way; only the `Choice` question's reading of context should keep it from being picked.

## Related

- `lessons/core-concepts/0002-choice.ipynb` — the `Choice` primitive, here constrained to candidates code already found rather than options you wrote by hand
- `cookbooks/extraction/date-extraction.ipynb` — a related extraction technique that reads structured *parts* instead of picking a whole span
- `cookbooks/extraction/sde-cascade.ipynb` — verifying a full extraction after the fact, rather than constraining the choice set up front

Questions? Ask your agent — it has the same research this cookbook was built from, in `research/typesafe-ai.md`.